In [14]:
# ============================================================
# ① データ準備コード 最新版
# 代表戦データ + 選手データの準備
# market_value_in_eur エラー対応
# 日本語フォント対応
# 重複列対策
# ============================================================

!pip install kagglehub japanize-matplotlib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import kagglehub
import os
import japanize_matplotlib

from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# グラフの日本語文字化け対策
plt.rcParams["axes.unicode_minus"] = False


# ----------------------------
# 国名の日本語変換
# ----------------------------

COUNTRY_JA = {
    "Japan": "日本",
    "Netherlands": "オランダ",
    "Sweden": "スウェーデン",
    "Tunisia": "チュニジア",
    "Brazil": "ブラジル",
    "Spain": "スペイン",
    "Italy": "イタリア",
    "France": "フランス",
    "Argentina": "アルゼンチン",
    "England": "イングランド",
    "Germany": "ドイツ",
    "Portugal": "ポルトガル",
    "Belgium": "ベルギー",
    "Croatia": "クロアチア",
    "United States": "アメリカ",
    "Korea, South": "韓国",
    "South Korea": "韓国",
    "Mexico": "メキシコ",
    "Morocco": "モロッコ",
    "Uruguay": "ウルグアイ",
    "Switzerland": "スイス",
    "Denmark": "デンマーク",
    "Serbia": "セルビア",
    "Poland": "ポーランド",
    "Norway": "ノルウェー",
    "Turkey": "トルコ",
    "Türkiye": "トルコ",
    "Australia": "オーストラリア",
    "Senegal": "セネガル",
    "Ghana": "ガーナ",
    "Saudi Arabia": "サウジアラビア",
    "Algeria": "アルジェリア",
    "Austria": "オーストリア",
    "Romania": "ルーマニア",
    "Czech Republic": "チェコ",
    "Nigeria": "ナイジェリア",
    "Cote d'Ivoire": "コートジボワール",
    "Côte d'Ivoire": "コートジボワール",
    "Ivory Coast": "コートジボワール",
    "Ireland": "アイルランド",
    "Cameroon": "カメルーン",
    "Bosnia-Herzegovina": "ボスニア・ヘルツェゴビナ",
    "Bosnia and Herzegovina": "ボスニア・ヘルツェゴビナ",
    "Slovakia": "スロバキア",
    "Albania": "アルバニア",
    "Mali": "マリ",
    "Slovenia": "スロベニア",
    "Wales": "ウェールズ",
    "DR Congo": "コンゴ民主共和国",
    "Congo DR": "コンゴ民主共和国",
    "Paraguay": "パラグアイ",
}


POSITION_JA = {
    "Goalkeeper": "GK",
    "Defender": "DF",
    "Midfield": "MF",
    "Attack": "FW",
    "Unknown": "不明",
}


def country_ja(team):
    return COUNTRY_JA.get(team, team)


def remove_duplicate_columns(df):
    return df.loc[:, ~df.columns.duplicated()].copy()


# ----------------------------
# 代表戦データ取得
# ----------------------------

def load_international_results():
    url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
    df = pd.read_csv(url)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "home_team", "away_team", "home_score", "away_score"])
    df = df.sort_values("date").reset_index(drop=True)

    def judge(row):
        if row["home_score"] > row["away_score"]:
            return 2  # ホーム勝ち
        elif row["home_score"] == row["away_score"]:
            return 1  # 引き分け
        else:
            return 0  # ホーム負け

    df["result"] = df.apply(judge, axis=1)

    return df


# ----------------------------
# 選手データ取得
# ----------------------------

def load_player_data():
    path = kagglehub.dataset_download("davidcariboo/player-scores")

    print("dataset path:", path)
    print("files:", os.listdir(path))

    players_file = os.path.join(path, "players.csv")
    appearances_file = os.path.join(path, "appearances.csv")
    valuations_file = os.path.join(path, "player_valuations.csv")

    players = pd.read_csv(players_file)
    appearances = pd.read_csv(appearances_file)
    valuations = pd.read_csv(valuations_file)

    print("\nplayers columns:")
    print(players.columns.tolist())

    print("\nappearances columns:")
    print(appearances.columns.tolist())

    print("\nvaluations columns:")
    print(valuations.columns.tolist())

    return players, appearances, valuations


# ----------------------------
# 選手特徴量作成
# ----------------------------

def build_player_features(players, appearances, valuations):
    players = remove_duplicate_columns(players.copy())
    appearances = remove_duplicate_columns(appearances.copy())
    valuations = remove_duplicate_columns(valuations.copy())

    # appearances の列名を安全に処理
    minutes_col = None
    for c in ["minutes_played", "minutes", "playing_time"]:
        if c in appearances.columns:
            minutes_col = c
            break

    goals_col = "goals" if "goals" in appearances.columns else None
    assists_col = "assists" if "assists" in appearances.columns else None

    agg_dict = {}

    if minutes_col is not None:
        agg_dict[minutes_col] = "sum"

    if goals_col is not None:
        agg_dict[goals_col] = "sum"

    if assists_col is not None:
        agg_dict[assists_col] = "sum"

    if len(agg_dict) == 0:
        app = pd.DataFrame({"player_id": players["player_id"]})
        app["minutes"] = 0
        app["goals"] = 0
        app["assists"] = 0
    else:
        app = appearances.groupby("player_id").agg(agg_dict).reset_index()

        rename_app = {}

        if minutes_col is not None:
            rename_app[minutes_col] = "minutes"

        if goals_col is not None:
            rename_app[goals_col] = "goals"

        if assists_col is not None:
            rename_app[assists_col] = "assists"

        app = app.rename(columns=rename_app)

        for col in ["minutes", "goals", "assists"]:
            if col not in app.columns:
                app[col] = 0

    # valuations の市場価値列を安全に処理
    value_candidates = [
        "market_value_in_eur",
        "market_value",
        "market_value_eur",
        "value",
    ]

    value_col = None

    for c in value_candidates:
        if c in valuations.columns:
            value_col = c
            break

    if value_col is None:
        print("valuations columns:", valuations.columns.tolist())
        raise ValueError("市場価値の列が見つかりません。valuations の列名を確認してください。")

    if "date" in valuations.columns:
        valuations["date"] = pd.to_datetime(valuations["date"], errors="coerce")

        latest_values = (
            valuations.sort_values("date")
            .groupby("player_id")
            .tail(1)[["player_id", value_col]]
        )
    else:
        latest_values = valuations.groupby("player_id").tail(1)[["player_id", value_col]]

    latest_values = latest_values.rename(columns={value_col: "market_value_in_eur"})

    # players と結合
    df = players.merge(app, on="player_id", how="left")
    df = df.merge(latest_values, on="player_id", how="left")

    df = remove_duplicate_columns(df)

    for col in ["minutes", "goals", "assists", "market_value_in_eur"]:
        if col not in df.columns:
            df[col] = 0
        df[col] = df[col].fillna(0)

    # 年齢
    if "date_of_birth" in df.columns:
        birth = pd.to_datetime(df["date_of_birth"], errors="coerce")
        df["age"] = 2026 - birth.dt.year
        df["age"] = df["age"].fillna(df["age"].median())
    else:
        df["age"] = 25

    # 列名統一
    rename_cols = {}

    if "name" in df.columns:
        rename_cols["name"] = "player_name"
    elif "pretty_name" in df.columns:
        rename_cols["pretty_name"] = "player_name"

    if "country_of_citizenship" in df.columns:
        rename_cols["country_of_citizenship"] = "national_team"
    elif "country" in df.columns:
        rename_cols["country"] = "national_team"

    if "sub_position" in df.columns:
        rename_cols["sub_position"] = "position"
    elif "position" not in df.columns and "main_position" in df.columns:
        rename_cols["main_position"] = "position"

    df = df.rename(columns=rename_cols)
    df = remove_duplicate_columns(df)

    # 必須列の補完
    if "player_name" not in df.columns:
        df["player_name"] = "Unknown Player"

    if "national_team" not in df.columns:
        df["national_team"] = "Unknown"

    if "position" not in df.columns:
        df["position"] = "Unknown"

    df["position"] = df["position"].fillna("Unknown").astype(str)
    df["national_team"] = df["national_team"].fillna("Unknown").astype(str)

    # 日本語列も追加
    df["national_team_ja"] = df["national_team"].map(COUNTRY_JA).fillna(df["national_team"])
    df["position_ja"] = df["position"].map(POSITION_JA).fillna(df["position"])

    return df


# ----------------------------
# 選手スコア作成
# ----------------------------

def safe_log_score(series, max_point):
    max_value = series.max()

    if max_value <= 0:
        return pd.Series(0, index=series.index)

    return np.log1p(series) / np.log1p(max_value) * max_point


def add_player_scores(df):
    df = remove_duplicate_columns(df.copy())

    df["market_value_in_eur"] = df["market_value_in_eur"].fillna(0)
    df["minutes"] = df["minutes"].fillna(0)
    df["goals"] = df["goals"].fillna(0)
    df["assists"] = df["assists"].fillna(0)

    value_score = safe_log_score(df["market_value_in_eur"], 40)
    minutes_score = safe_log_score(df["minutes"], 25)
    goal_score = safe_log_score(df["goals"], 20)
    assist_score = safe_log_score(df["assists"], 15)

    df["player_score"] = value_score + minutes_score + goal_score + assist_score

    return df


# ----------------------------
# 実行
# ----------------------------

intl = load_international_results()
players, appearances, valuations = load_player_data()

players_latest = build_player_features(players, appearances, valuations)
players_scored = add_player_scores(players_latest)

players_scored = remove_duplicate_columns(players_scored)

print("==================================================")
print("データ準備完了")
print("==================================================")
print("代表戦データ:", intl.shape)
print("代表戦データの最新日:", intl["date"].max())
print("選手データ:", players_scored.shape)

display(players_scored[[
    "player_name",
    "national_team",
    "national_team_ja",
    "position",
    "position_ja",
    "age",
    "market_value_in_eur",
    "minutes",
    "goals",
    "assists",
    "player_score"
]].head())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 35.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Using Colab cache for faster access to the 'player-scores' dataset.
dataset path: /kaggle/input/player-scores
files: ['players.csv', 'national_teams.csv', 'countries.csv', 'competitions.csv', 'games.csv', 'transfers.csv', 'game_events.csv', 'club_games.csv', 'player_valuations.csv', 'game_lineups.csv', 'appearances.csv', 'clubs.csv']

players columns:
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

appearances columns:
['appearance_id', 'game_

,player_name,national_team,national_team_ja,position,position_ja,age,market_value_in_eur,minutes,goals,assists,player_score
0,Miroslav Klose,Germany,ドイツ,Centre-Forward,Centre-Forward,48.0,0,8808.0,48.0,25.0,42.268315
1,Roman Weidenfeller,Germany,ドイツ,Goalkeeper,GK,46.0,0,13508.0,0.0,0.0,21.813455
2,Dimitar Berbatov,Bulgaria,Bulgaria,Centre-Forward,Centre-Forward,45.0,0,8788.0,38.0,13.0,39.820678
3,Lúcio,Brazil,ブラジル,Centre-Back,Centre-Back,48.0,0,307.0,0.0,0.0,13.141816
4,Tom Starke,Germany,ドイツ,Goalkeeper,GK,45.0,0,1080.0,0.0,0.0,16.021365


In [ ]:
# ============================================================
# ② AIモデルコード
# Elo + 直近5試合 + 直近10試合で勝敗予測AIを作る
# ============================================================

# ----------------------------
# 直近成績
# ----------------------------

def get_recent_stats(history, team, date, n=10):
    past = history[
        ((history["home_team"] == team) | (history["away_team"] == team)) &
        (history["date"] < date)
    ].tail(n)

    if len(past) == 0:
        return {
            "win_rate": 0.33,
            "gf_avg": 1.0,
            "ga_avg": 1.0
        }

    wins = 0
    gf = []
    ga = []

    for _, r in past.iterrows():
        if r["home_team"] == team:
            team_goals = r["home_score"]
            opp_goals = r["away_score"]
        else:
            team_goals = r["away_score"]
            opp_goals = r["home_score"]

        gf.append(team_goals)
        ga.append(opp_goals)

        if team_goals > opp_goals:
            wins += 1

    return {
        "win_rate": wins / len(past),
        "gf_avg": np.mean(gf),
        "ga_avg": np.mean(ga)
    }


# ----------------------------
# Eloレーティング
# ----------------------------

def elo_expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def add_elo_features(intl, base_elo=1500, k=30, home_advantage=60):
    df = intl.copy().sort_values("date").reset_index(drop=True)
    ratings = defaultdict(lambda: base_elo)

    home_elo_list = []
    away_elo_list = []
    elo_diff_list = []
    home_expected_list = []
    away_expected_list = []

    for _, row in df.iterrows():
        home = row["home_team"]
        away = row["away_team"]

        home_elo = ratings[home]
        away_elo = ratings[away]

        neutral = int(row["neutral"]) if "neutral" in row else 0
        home_adv = 0 if neutral == 1 else home_advantage

        adjusted_home_elo = home_elo + home_adv

        home_expected = elo_expected_score(adjusted_home_elo, away_elo)
        away_expected = 1 - home_expected

        home_elo_list.append(home_elo)
        away_elo_list.append(away_elo)
        elo_diff_list.append(adjusted_home_elo - away_elo)
        home_expected_list.append(home_expected)
        away_expected_list.append(away_expected)

        if row["home_score"] > row["away_score"]:
            home_actual = 1.0
            away_actual = 0.0
        elif row["home_score"] == row["away_score"]:
            home_actual = 0.5
            away_actual = 0.5
        else:
            home_actual = 0.0
            away_actual = 1.0

        ratings[home] = home_elo + k * (home_actual - home_expected)
        ratings[away] = away_elo + k * (away_actual - away_expected)

    df["home_elo"] = home_elo_list
    df["away_elo"] = away_elo_list
    df["elo_diff"] = elo_diff_list
    df["home_expected_by_elo"] = home_expected_list
    df["away_expected_by_elo"] = away_expected_list

    return df


# ----------------------------
# 機械学習用データ作成
# ----------------------------

def build_ml_dataset_with_elo(intl, since="2023-01-01"):
    elo_df = add_elo_features(intl)
    base = elo_df[elo_df["date"] >= since].copy()

    rows = []

    for _, row in base.iterrows():
        date = row["date"]
        home = row["home_team"]
        away = row["away_team"]

        hist = intl[intl["date"] < date]

        hs10 = get_recent_stats(hist, home, date, n=10)
        aw10 = get_recent_stats(hist, away, date, n=10)

        hs5 = get_recent_stats(hist, home, date, n=5)
        aw5 = get_recent_stats(hist, away, date, n=5)

        rows.append({
            "date": date,
            "home_team": home,
            "away_team": away,

            "home_win_rate_10": hs10["win_rate"],
            "away_win_rate_10": aw10["win_rate"],
            "home_gf_avg_10": hs10["gf_avg"],
            "away_gf_avg_10": aw10["gf_avg"],
            "home_ga_avg_10": hs10["ga_avg"],
            "away_ga_avg_10": aw10["ga_avg"],

            "home_win_rate_5": hs5["win_rate"],
            "away_win_rate_5": aw5["win_rate"],
            "home_gf_avg_5": hs5["gf_avg"],
            "away_gf_avg_5": aw5["gf_avg"],
            "home_ga_avg_5": hs5["ga_avg"],
            "away_ga_avg_5": aw5["ga_avg"],

            "win_rate_diff_10": hs10["win_rate"] - aw10["win_rate"],
            "gf_diff_10": hs10["gf_avg"] - aw10["gf_avg"],
            "ga_diff_10": hs10["ga_avg"] - aw10["ga_avg"],

            "win_rate_diff_5": hs5["win_rate"] - aw5["win_rate"],
            "gf_diff_5": hs5["gf_avg"] - aw5["gf_avg"],
            "ga_diff_5": hs5["ga_avg"] - aw5["ga_avg"],

            "home_elo": row["home_elo"],
            "away_elo": row["away_elo"],
            "elo_diff": row["elo_diff"],
            "home_expected_by_elo": row["home_expected_by_elo"],
            "away_expected_by_elo": row["away_expected_by_elo"],

            "neutral": int(row["neutral"]) if "neutral" in row else 0,
            "result": row["result"]
        })

    return pd.DataFrame(rows)


# ----------------------------
# モデル学習
# ----------------------------

ml_df_elo = build_ml_dataset_with_elo(intl, since="2023-01-01")

features_elo = [
    "home_win_rate_10",
    "away_win_rate_10",
    "home_gf_avg_10",
    "away_gf_avg_10",
    "home_ga_avg_10",
    "away_ga_avg_10",

    "home_win_rate_5",
    "away_win_rate_5",
    "home_gf_avg_5",
    "away_gf_avg_5",
    "home_ga_avg_5",
    "away_ga_avg_5",

    "win_rate_diff_10",
    "gf_diff_10",
    "ga_diff_10",

    "win_rate_diff_5",
    "gf_diff_5",
    "ga_diff_5",

    "home_elo",
    "away_elo",
    "elo_diff",
    "home_expected_by_elo",
    "away_expected_by_elo",

    "neutral"
]

X = ml_df_elo[features_elo]
y = ml_df_elo["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_elo = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_elo.fit(X_train, y_train)

pred_test_elo = model_elo.predict(X_test)
acc_test_elo = accuracy_score(y_test, pred_test_elo)

print("==================================================")
print("AIモデル精度")
print("==================================================")
print("採用モデル：Elo + 直近5試合 + 直近10試合")
print(f"テストデータでの正解率：{round(acc_test_elo * 100, 2)}%")

print("\n分類レポート")
print(classification_report(
    y_test,
    pred_test_elo,
    target_names=["ホーム負け", "引き分け", "ホーム勝ち"]
))

print("\n混同行列")
print(confusion_matrix(y_test, pred_test_elo))

In [ ]:
# ============================================================
# ③ 試合シミュレーションコード 修正版
# チーム力・スコア分布・グループ突破率・試合ログ生成
# duplicate labels エラー対応済み
# ============================================================

# ----------------------------
# 重複列を削除する関数
# ----------------------------

def remove_duplicate_columns(df):
    return df.loc[:, ~df.columns.duplicated()].copy()


# ----------------------------
# チーム力
# ----------------------------

def team_squad(players_df, team_name, top_n=26):
    players_df = remove_duplicate_columns(players_df)

    team = players_df[
        players_df["national_team"].astype(str).str.lower() == team_name.lower()
    ].copy()

    team = remove_duplicate_columns(team)
    team = team.sort_values("player_score", ascending=False)

    return team.head(top_n)


def calculate_team_strength(players_df, team_name, verbose=False):
    players_df = remove_duplicate_columns(players_df)
    squad = team_squad(players_df, team_name, top_n=26)

    if len(squad) == 0:
        return {
            "team": team_name,
            "squad_size": 0,
            "total_strength": 0,
            "starting_strength": 0,
            "bench_strength": 0,
            "squad_market_value": 0
        }

    squad = remove_duplicate_columns(squad)

    starting = squad.head(11)
    bench = squad.iloc[11:]

    result = {
        "team": team_name,
        "squad_size": len(squad),
        "total_strength": squad["player_score"].sum(),
        "starting_strength": starting["player_score"].sum(),
        "bench_strength": bench["player_score"].sum(),
        "squad_market_value": squad["market_value_in_eur"].sum()
    }

    if verbose:
        print(result)

        display_cols = [
            "player_name",
            "national_team",
            "position",
            "age",
            "market_value_in_eur",
            "minutes",
            "goals",
            "assists",
            "player_score"
        ]

        display_cols = [c for c in display_cols if c in starting.columns]
        display(starting[display_cols])

    return result


# ----------------------------
# 期待得点
# ----------------------------

def expected_goals(strength_a, strength_b, base_goal=1.30, scale=900):
    diff = strength_a - strength_b

    xg_a = base_goal + diff / scale
    xg_b = base_goal - diff / scale

    xg_a = max(0.2, xg_a)
    xg_b = max(0.2, xg_b)

    return xg_a, xg_b


# ----------------------------
# 試合を1万回シミュレーション
# ----------------------------

def simulate_match_many_fast(team_a, team_b, players_df, n=10000, seed=42):
    players_df = remove_duplicate_columns(players_df)
    np.random.seed(seed)

    strength_a = calculate_team_strength(players_df, team_a)["total_strength"]
    strength_b = calculate_team_strength(players_df, team_b)["total_strength"]

    xg_a, xg_b = expected_goals(strength_a, strength_b)

    goals_a = np.random.poisson(xg_a, n)
    goals_b = np.random.poisson(xg_b, n)

    a_win = np.sum(goals_a > goals_b)
    draw = np.sum(goals_a == goals_b)
    b_win = np.sum(goals_a < goals_b)

    scores = {}

    for ga, gb in zip(goals_a, goals_b):
        score = f"{ga}-{gb}"
        scores[score] = scores.get(score, 0) + 1

    return {
        "team_a": team_a,
        "team_b": team_b,
        "xg_a": round(xg_a, 2),
        "xg_b": round(xg_b, 2),
        "team_a_win_rate": round(a_win / n * 100, 2),
        "draw_rate": round(draw / n * 100, 2),
        "team_b_win_rate": round(b_win / n * 100, 2),
        "most_common_score": max(scores, key=scores.get),
        "score_distribution": scores
    }


def show_match_result_ja(result):
    team_a = result["team_a"]
    team_b = result["team_b"]

    team_a_ja = country_ja(team_a)
    team_b_ja = country_ja(team_b)

    summary = pd.DataFrame([{
        "対戦カード": f"{team_a_ja} vs {team_b_ja}",
        f"{team_a_ja}勝利": f"{result['team_a_win_rate']}%",
        "引き分け": f"{result['draw_rate']}%",
        f"{team_b_ja}勝利": f"{result['team_b_win_rate']}%",
        f"{team_a_ja}期待得点": result["xg_a"],
        f"{team_b_ja}期待得点": result["xg_b"],
        "最頻スコア": result["most_common_score"]
    }])

    print("==================================================")
    print("試合シミュレーション結果")
    print("==================================================")
    display(summary)

    scores = pd.DataFrame([
        {
            "スコア": score,
            "回数": count,
            "確率": round(count / sum(result["score_distribution"].values()) * 100, 2)
        }
        for score, count in result["score_distribution"].items()
    ])

    scores = scores.sort_values("回数", ascending=False).head(12)

    print("\n出やすいスコア TOP12")
    display(scores)

    plt.figure(figsize=(10, 5))
    plt.bar(scores["スコア"], scores["回数"])
    plt.title(f"{team_a_ja} vs {team_b_ja} スコア分布")
    plt.xlabel("スコア")
    plt.ylabel("出現回数")
    plt.xticks(rotation=45)
    plt.show()


# ----------------------------
# グループリーグシミュレーション
# ----------------------------

def simulate_group_many_fast(group_teams, players_df, n=10000, seed=42):
    players_df = remove_duplicate_columns(players_df)
    np.random.seed(seed)

    strengths = {
        team: calculate_team_strength(players_df, team)["total_strength"]
        for team in group_teams
    }

    fixtures = []

    for i in range(len(group_teams)):
        for j in range(i + 1, len(group_teams)):
            fixtures.append((group_teams[i], group_teams[j]))

    first_count = {team: 0 for team in group_teams}
    advance_count = {team: 0 for team in group_teams}
    points_list = {team: [] for team in group_teams}

    for _ in range(n):
        table = {
            team: {
                "pts": 0,
                "gf": 0,
                "ga": 0,
                "gd": 0
            }
            for team in group_teams
        }

        for team_a, team_b in fixtures:
            xg_a, xg_b = expected_goals(
                strengths[team_a],
                strengths[team_b]
            )

            goals_a = np.random.poisson(xg_a)
            goals_b = np.random.poisson(xg_b)

            table[team_a]["gf"] += goals_a
            table[team_a]["ga"] += goals_b
            table[team_b]["gf"] += goals_b
            table[team_b]["ga"] += goals_a

            if goals_a > goals_b:
                table[team_a]["pts"] += 3
            elif goals_a == goals_b:
                table[team_a]["pts"] += 1
                table[team_b]["pts"] += 1
            else:
                table[team_b]["pts"] += 3

        for team in group_teams:
            table[team]["gd"] = table[team]["gf"] - table[team]["ga"]

        ranking = sorted(
            group_teams,
            key=lambda t: (
                table[t]["pts"],
                table[t]["gd"],
                table[t]["gf"]
            ),
            reverse=True
        )

        first_count[ranking[0]] += 1

        for team in ranking[:2]:
            advance_count[team] += 1

        for team in group_teams:
            points_list[team].append(table[team]["pts"])

    result = []

    for team in group_teams:
        result.append({
            "team": team,
            "国": country_ja(team),
            "first_rate": round(first_count[team] / n * 100, 2),
            "advance_rate": round(advance_count[team] / n * 100, 2),
            "avg_points": round(np.mean(points_list[team]), 2)
        })

    return pd.DataFrame(result).sort_values("advance_rate", ascending=False)


def show_group_result_ja(group_result):
    df = group_result.copy()

    display_df = df[[
        "国",
        "first_rate",
        "advance_rate",
        "avg_points"
    ]].rename(columns={
        "first_rate": "1位確率",
        "advance_rate": "突破確率",
        "avg_points": "平均勝ち点"
    })

    print("==================================================")
    print("グループリーグ突破確率")
    print("==================================================")
    display(display_df)

    plt.figure(figsize=(8, 5))
    plt.bar(display_df["国"], display_df["突破確率"])
    plt.title("グループリーグ突破確率")
    plt.ylabel("突破確率（%）")
    plt.ylim(0, 100)
    plt.show()


# ----------------------------
# 1分ごとの試合ログ生成AI 修正版
# ----------------------------

def normalize_position(pos):
    pos = str(pos)

    if pos in ["Goalkeeper", "Keeper", "GK"]:
        return "Goalkeeper"

    if pos in ["Defender", "Centre-Back", "Left-Back", "Right-Back", "DF"]:
        return "Defender"

    if pos in [
        "Midfield",
        "Central Midfield",
        "Defensive Midfield",
        "Attacking Midfield",
        "Left Midfield",
        "Right Midfield",
        "MF"
    ]:
        return "Midfield"

    if pos in [
        "Attack",
        "Centre-Forward",
        "Second Striker",
        "Left Winger",
        "Right Winger",
        "FW"
    ]:
        return "Attack"

    return "Unknown"


def get_team_players_for_sim(players_df, team_name, top_n=11):
    players_df = remove_duplicate_columns(players_df)

    squad = team_squad(players_df, team_name, top_n=26)

    if len(squad) == 0:
        return []

    squad = remove_duplicate_columns(squad)

    if "position" not in squad.columns:
        squad["position"] = "Unknown"

    squad["position"] = squad["position"].fillna("Unknown").astype(str)
    squad["position_group"] = squad["position"].apply(normalize_position)

    gk = squad[squad["position_group"] == "Goalkeeper"].head(1)
    df = squad[squad["position_group"] == "Defender"].head(4)
    mf = squad[squad["position_group"] == "Midfield"].head(3)
    fw = squad[squad["position_group"] == "Attack"].head(3)

    selected = pd.concat([gk, df, mf, fw])

    if len(selected) < top_n:
        already = set(selected["player_name"])
        extra = squad[~squad["player_name"].isin(already)].head(top_n - len(selected))
        selected = pd.concat([selected, extra])

    return selected.head(top_n).to_dict("records")


def weighted_choice_players(players):
    if len(players) == 0:
        return None

    scores = np.array([
        max(float(p.get("player_score", 1)), 1)
        for p in players
    ])

    if scores.sum() <= 0:
        probs = np.ones(len(players)) / len(players)
    else:
        probs = scores / scores.sum()

    idx = np.random.choice(len(players), p=probs)

    return players[idx]


def simulate_live_match(team_a, team_b, players_df, minutes=90, seed=42):
    players_df = remove_duplicate_columns(players_df)

    random.seed(seed)
    np.random.seed(seed)

    players_a = get_team_players_for_sim(players_df, team_a)
    players_b = get_team_players_for_sim(players_df, team_b)

    if len(players_a) == 0 or len(players_b) == 0:
        return {
            "team_a": team_a,
            "team_b": team_b,
            "team_a_score": 0,
            "team_b_score": 0,
            "result": "選手データ不足",
            "log": ["選手データが不足しているため、試合ログを生成できませんでした。"]
        }

    strength_a = calculate_team_strength(players_df, team_a)["total_strength"]
    strength_b = calculate_team_strength(players_df, team_b)["total_strength"]

    xg_a, xg_b = expected_goals(strength_a, strength_b)

    goal_rate_a = xg_a / minutes
    goal_rate_b = xg_b / minutes

    score_a = 0
    score_b = 0
    log = []

    team_a_ja = country_ja(team_a)
    team_b_ja = country_ja(team_b)

    log.append(f"0分　試合開始：{team_a_ja} vs {team_b_ja}")

    if strength_a + strength_b <= 0:
        possession_a = 0.5
    else:
        possession_a = strength_a / (strength_a + strength_b)

    for minute in range(1, minutes + 1):
        attacking_team = team_a if random.random() < possession_a else team_b

        if attacking_team == team_a:
            attack_players = players_a
            team_ja = team_a_ja
            opponent_ja = team_b_ja
            goal_rate = goal_rate_a
        else:
            attack_players = players_b
            team_ja = team_b_ja
            opponent_ja = team_a_ja
            goal_rate = goal_rate_b

        actor = weighted_choice_players(attack_players)

        if actor is None:
            continue

        actor_name = actor.get("player_name", "Unknown Player")
        actor_score = float(actor.get("player_score", 1))

        # 何も起きない時間帯を作る
        if random.random() > 0.18:
            continue

        event_roll = random.random()

        if event_roll < 0.40:
            log.append(f"{minute}分　{team_ja}：{actor_name} がボールをつなぐ")

        elif event_roll < 0.65:
            log.append(f"{minute}分　{team_ja}：{actor_name} が前線へパス")

        elif event_roll < 0.83:
            log.append(f"{minute}分　{team_ja}：{actor_name} がチャンスを作る")

        else:
            shoot_bonus = max(actor_score, 1) / 100
            goal_probability = goal_rate * 5 * shoot_bonus

            if random.random() < goal_probability:
                if attacking_team == team_a:
                    score_a += 1
                else:
                    score_b += 1

                log.append(
                    f"{minute}分　ゴール！ {team_ja}：{actor_name} が決めた！ "
                    f"現在 {team_a_ja} {score_a}-{score_b} {team_b_ja}"
                )
            else:
                log.append(
                    f"{minute}分　{team_ja}：{actor_name} がシュート！"
                    f"しかし{opponent_ja}が守る"
                )

    log.append(f"90分　試合終了：{team_a_ja} {score_a}-{score_b} {team_b_ja}")

    if score_a > score_b:
        result = f"{team_a_ja}の勝利"
    elif score_a < score_b:
        result = f"{team_b_ja}の勝利"
    else:
        result = "引き分け"

    return {
        "team_a": team_a,
        "team_b": team_b,
        "team_a_score": score_a,
        "team_b_score": score_b,
        "result": result,
        "log": log
    }


def show_live_match_result(match):
    for line in match["log"]:
        print(line)

    print("\n==============================")
    print("最終結果")
    print("==============================")
    print(
        f"{country_ja(match['team_a'])} "
        f"{match['team_a_score']} - {match['team_b_score']} "
        f"{country_ja(match['team_b'])}"
    )
    print(match["result"])

In [ ]:
TEAM_A = "Japan"
TEAM_B = "Netherlands"
GROUP_TEAMS = ["Japan", "Netherlands", "Sweden", "Tunisia"]

match_result = simulate_match_many_fast(
    TEAM_A,
    TEAM_B,
    players_scored,
    n=10000,
    seed=42
)

show_match_result_ja(match_result)

group_result = simulate_group_many_fast(
    GROUP_TEAMS,
    players_scored,
    n=10000,
    seed=42
)

show_group_result_ja(group_result)

live_match = simulate_live_match(
    TEAM_A,
    TEAM_B,
    players_scored,
    minutes=90,
    seed=7
)

show_live_match_result(live_match)